In [ ]:
from pathlib import Path

import numpy as np
from jaxtyping import Float
import polars as pl
import matplotlib.pyplot as plt

# scipy
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

# muutils
from muutils.jsonlines import jsonl_write, jsonl_load
from muutils.dbg import dbg, dbg_tensor
from muutils.tensor_info import array_summary

# attention-motifs
from attention_motifs.bins import Bins

# pl.set_option('display.max_rows', 200)
# pl.set_option('display.max_columns', 200)


In [ ]:
# DATA: pl.DataFrame = pl.DataFrame(jsonl_load("../data/scalar_features.jsonl"))

DATA: pl.DataFrame = pl.read_ndjson(Path("../data/scalar_features_fixed.jsonl"))

In [ ]:
DATA.head()

In [ ]:
DATA.describe()

In [ ]:
def nan_stats(data: pl.DataFrame) -> pl.DataFrame:
    return pl.DataFrame({
        'count': data.isna().sum(),
        'percentage': (data.isna().sum() / len(data) * 100).round(2)
    }).sort_values('count', ascending=False)

nan_stats(DATA).head()

In [ ]:
# all cols except
nan_cols = [
	"feat.markov_transition.time.kurtosis",
	"feat.markov_transition.time.skewness",
]
# because those have nan
DATA_NANLESS: pl.DataFrame = DATA.drop(nan_cols, axis=1)
nan_stats(DATA_NANLESS).head()

In [ ]:


def preprocess_data(df: pl.DataFrame, feature_cols: list[str]) -> pl.DataFrame:
    """Standardize selected features."""
    scaler: StandardScaler = StandardScaler()
    arr: np.ndarray = scaler.fit_transform(df[feature_cols])
    df_scaled: pl.DataFrame = pl.DataFrame(arr, columns=feature_cols, index=df.index)
    return df_scaled

def apply_pca(data: pl.DataFrame, n_components: int) -> np.ndarray:
    """Compute PCA."""
    pca: PCA = PCA(n_components=n_components, random_state=0)
    reduced: np.ndarray = pca.fit_transform(data)
    return reduced

def apply_tsne(data: pl.DataFrame, n_components: int = 2, perplexity: float = 30.0) -> np.ndarray:
    """Compute t-SNE."""
    tsne: TSNE = TSNE(n_components=n_components, perplexity=perplexity, random_state=0)
    embedded: np.ndarray = tsne.fit_transform(data)
    return embedded

def cluster_kmeans(data: np.ndarray, n_clusters: int) -> np.ndarray:
    """Cluster with K-Means."""
    kmeans: KMeans = KMeans(n_clusters=n_clusters, random_state=0)
    labels: np.ndarray = kmeans.fit_predict(data)
    return labels

def plot_correlation_matrix(df: pl.DataFrame, feature_cols: List[str]) -> None:
    """Plot a correlation matrix of features and identify features with NaN correlations."""
    # Calculate correlation matrix
    corr_df: pl.DataFrame = df[feature_cols].corr()
    corr_mat: np.ndarray = corr_df.values
    dbg_tensor(corr_mat)  # Keep the original debugging call
    
    # Find features with NaN values in their correlations
    nan_features = []
    for i, feature in enumerate(feature_cols):
        if np.isnan(corr_mat[i]).all():
            nan_features.append(feature)
            print(f"Feature with NaN correlations: {feature}\n\t{array_summary(df[feature].to_numpy(dtype=float))}\n\t{array_summary(corr_mat[i])}")
        
    # Plot
    fig: plt.Figure = plt.figure()
    ax: plt.Axes = fig.add_subplot(111)
    cax: plt.AxesImage = ax.imshow(corr_mat, aspect='auto')
    plt.colorbar(cax)
    ax.set_title("Feature Correlation Matrix")
    plt.tight_layout()
    plt.show()

def plot_embedding(embedding: np.ndarray, labels: pl.Series, title: str) -> None:
    """Scatter plot of 2D embedding with label text."""
    fig: plt.Figure = plt.figure()
    ax: plt.Axes = fig.add_subplot(111)
    sc: plt.PathCollection = ax.scatter(embedding[:, 0], embedding[:, 1])
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.set_xlabel("Dim 1")
    ax.set_ylabel("Dim 2")
    # Optionally, overlay text for each point
    # for i, lbl in enumerate(labels):
    #     ax.text(embedding[i, 0], embedding[i, 1], lbl, fontsize=6)
    plt.show()

def main_example(df: pl.DataFrame) -> None:
    # Pick feature columns
    feature_cols: list[str] = [col for col in df.columns if col.startswith("feat.")]
    
    # Plot correlation matrix
    plot_correlation_matrix(df, feature_cols)
    
    # Scale
    df_scaled: pl.DataFrame = preprocess_data(df, feature_cols)
    display(nan_stats(df_scaled).head())
    
    # PCA
    pca_data: np.ndarray = apply_pca(df_scaled, n_components=5)
    
    # t-SNE
    tsne_data: np.ndarray = apply_tsne(df_scaled)
    
    # Clustering
    kmeans_labels: np.ndarray = cluster_kmeans(tsne_data, n_clusters=5)
    
    # Plot PCA embedding
    # (using first two PCA components for a 2D plot)
    plot_embedding(pca_data[:, :2], df["activation.cls"], "PCA of Features")
    
    # Plot t-SNE embedding
    plot_embedding(tsne_data, df["activation.cls"], "t-SNE of Features")
    
    # Plot the clusters on t-SNE
    # (If you prefer numeric cluster labels, you can cast them to string.)
    cluster_series: pl.Series = pl.Series(kmeans_labels, index=df.index, dtype=str)
    plot_embedding(tsne_data, cluster_series, "K-Means Clusters (t-SNE)")


main_example(DATA_NANLESS)